##### Using weekofyear()

- To **Extract** the **week number (1-53)** of a given **date or timestamp** as **integer**.
- Year boundary cases:
  - `Jan 1 might belong to last year's week`.
  - `Dec 31 might belong to next year's week`.

##### Syntax

     weekofyear(expr)

|  Argument   |           Description                                      |
|-------------|------------------------------------------------------------|
|   expr      | The **input column** must be a **date or timestamp type**. |
|  Return     | An **integer**                                             |

**ISO 8601 Compliance:**
- The function follows the **ISO 8601 standard** week numbering.
- A **week** is considered to **start on a Monday** and **week 1** is the **first week with more than 3 days**, as defined by `ISO 8601`.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, to_date, weekofyear, current_timestamp, to_timestamp

##### 1) Extracting Week of Year from a Date Column

In [0]:
# Create a sample DataFrame with a date column (ensure the column is a date type)
df_wk = spark.createDataFrame([
    (1, "2021-12-30"),
    (2, "2021-12-31"),
    (3, "2022-01-01"),
    (4, "2022-01-02"),
    (5, "2022-01-03"),
    (6, "2022-12-30"),
    (7, "2022-12-31"),
    (8, "2023-01-01"),
    (9, "2023-01-02"),
    (10, "2023-12-30"),
    (11, "2023-12-31"),
    (12, "2024-01-01"),
    (13, "2024-01-02"),
    (14, "2024-12-29"),
    (15, "2024-12-30"),
    (16, "2024-12-31"),
    (17, "2025-01-01"),
    (18, "2025-12-28"),
    (19, "2025-12-29"),
    (20, "2025-12-30"),
    (21, "2025-12-31"),
    (22, "2026-01-03"),
    (23, "2026-03-10")
], ["id", "date_str"])

# Cast the string column to a date type
df_wk = df_wk.withColumn('date', F.to_date(F.col('date_str')))

# Use weekofyear() to add a new column with the week number
df_wk.withColumn(
    'week_number', 
    F.weekofyear(F.col('date'))
).display()

id,date_str,date,week_number
1,2021-12-30,2021-12-30,52
2,2021-12-31,2021-12-31,52
3,2022-01-01,2022-01-01,52
4,2022-01-02,2022-01-02,52
5,2022-01-03,2022-01-03,1
6,2022-12-30,2022-12-30,52
7,2022-12-31,2022-12-31,52
8,2023-01-01,2023-01-01,52
9,2023-01-02,2023-01-02,1
10,2023-12-30,2023-12-30,52


In [0]:
df_wk_final = spark.range(1) \
    .select(current_timestamp().alias("current_time")) \
    .select("current_time", weekofyear("current_time").alias("week_number"))

display(df_wk_final)

current_time,week_number
2026-03-12T07:35:14.401Z,11


##### 2) Extracting Week of Year from a Timestamp Column

In [0]:
# Create a DataFrame with a timestamp column
data = [("2022-07-25 12:00:00",), ("2023-01-01 10:00:00",), ("2022-12-31 23:59:59",), ("2023-01-01 00:00:00",)]
df_ts = spark.createDataFrame(data, ["timestamp"])

# Convert the string timestamp column to a TimestampType
df_ts = df_ts.withColumn("timestamp", to_timestamp("timestamp"))
display(df_ts)

timestamp
2022-07-25T12:00:00.000Z
2023-01-01T10:00:00.000Z
2022-12-31T23:59:59.000Z
2023-01-01T00:00:00.000Z


In [0]:
# Extract the week of year from the timestamp column
df_ts_wk = df_ts.withColumn("week_of_year", weekofyear(col("timestamp")))

# Display the result
display(df_ts_wk)

timestamp,week_of_year
2022-07-25T12:00:00.000Z,30
2023-01-01T10:00:00.000Z,52
2022-12-31T23:59:59.000Z,52
2023-01-01T00:00:00.000Z,52


##### 3) Using weekofyear() with Filter

In [0]:
# Create a DataFrame with a date column
data = [("2022-07-25",), ("2023-01-01",), ("2023-01-02",), ("2023-01-03",), ("2022-12-31",)]
df_dt = spark.createDataFrame(data, ["date"])

# Convert the date column to a DateType
df_dt = df_dt.withColumn("date", to_date("date"))
display(df_dt)

date
2022-07-25
2023-01-01
2023-01-02
2023-01-03
2022-12-31


In [0]:
# Extract the week of year from the date column and filter for weeks 30-52
df_dt_wk = df_dt.withColumn("week_of_year", weekofyear(col("date")))
display(df_dt_wk)

date,week_of_year
2022-07-25,30
2023-01-01,52
2023-01-02,1
2023-01-03,1
2022-12-31,52


In [0]:
df_dt_wk_filter = df_dt_wk.filter(col("week_of_year").between(30, 52))

# Display the result
display(df_dt_wk_filter)

date,week_of_year
2022-07-25,30
2023-01-01,52
2022-12-31,52


##### 4) How to Group by Week in PySpark?

In [0]:
# define data
data = [['2023-04-11', 200],
        ['2023-04-15', 150],
        ['2023-04-17', 120],
        ['2023-05-21', 170],
        ['2023-05-23', 300],
        ['2023-10-26', 450],
        ['2023-10-28', 320],
        ['2023-10-29', 470]]
  
# define column names
columns = ['start_date', 'sales']
  
# create dataframe using data and column names
df_grp = spark.createDataFrame(data, columns) 
  
# view dataframe
display(df_grp)

start_date,sales
2023-04-11,200
2023-04-15,150
2023-04-17,120
2023-05-21,170
2023-05-23,300
2023-10-26,450
2023-10-28,320
2023-10-29,470


In [0]:
df_grp_wk = df_grp.select("start_date", "sales", weekofyear("start_date").alias("week_number"))
display(df_grp_wk)

start_date,sales,week_number
2023-04-11,200,15
2023-04-15,150,15
2023-04-17,120,16
2023-05-21,170,20
2023-05-23,300,21
2023-10-26,450,43
2023-10-28,320,43
2023-10-29,470,43


In [0]:
from pyspark.sql.functions import weekofyear, sum, count

# calculate sum of sales by week
sum_count = df_grp \
    .groupBy(weekofyear('start_date').alias('sum_week_number')) \
    .agg(sum('sales').alias('sum_sales'),
         count('sales').alias('cnt_sales'))
    
display(sum_count)

sum_week_number,sum_sales,cnt_sales
15,350,2
16,120,1
20,170,1
21,300,1
43,1240,3


- The sum of sales for the `15th week` of the year was `350`.
- The sum of sales for the `16th week` of the year was `120`.
- The sum of sales for the `20th week` of the year was `170`.
- The sum of sales for the `21st week` of the year was `300`.
- The sum of sales for the `43rd week` of the year was `1240`.